# NLP Assignment 3 - Part 2: Recurrent NMT

BiLSTM encoder, unidirectional LSTM decoder, and additive attention for French -> English translation.

## 0. Imports

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'nmt_transformer'))

import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datasets import load_from_disk

from config import *
from data import get_dataloaders
from recurrent_model import RecurrentNMT
from recurrent_inference import lstm_greedy_decode, lstm_beam_search
from train import train, save_checkpoint, load_checkpoint
from utils import compute_bleu_dataset, ids_to_tokens, plot_attention, plot_loss_curves

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEAM_SIZE = 4

def clean(text):
    return ' '.join(text.replace('▁', ' ').split())

print(f'Device: {DEVICE}')

Device: cuda


## 1. Load Data

In [2]:
train_loader, val_loader, test_loader, src_tokenizer, tgt_tokenizer = get_dataloaders(
    DATA_PATH, TOKENIZER_FR_PATH, TOKENIZER_EN_PATH,
    batch_size=LSTM_BATCH_SIZE, max_seq_len=MAX_SEQ_LEN,
    bos_id=BOS_ID, eos_id=EOS_ID, pad_id=PAD_ID,
)
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

ds = load_from_disk(DATA_PATH)
sample = ds['train'][0]
print(f"Sample FR: {sample['text_fr']}")
print(f"Sample EN: {sample['text_en']}")

Train batches : 272
Val   batches : 16
Test  batches : 16
Sample FR: je suis dure .
Sample EN: i m tough .


## 2. Build Recurrent Model

In [3]:
model = RecurrentNMT(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    embed_size=LSTM_EMBED_SIZE,
    hidden_size=LSTM_HIDDEN_SIZE,
    num_layers=LSTM_NUM_LAYERS,
    dropout=LSTM_DROPOUT,
    pad_id=PAD_ID,
).to(DEVICE)

print(f'Embedding size: {LSTM_EMBED_SIZE}')
print(f'Hidden size   : {LSTM_HIDDEN_SIZE}')
print(f'Num layers    : {LSTM_NUM_LAYERS}')
print(f'Dropout       : {LSTM_DROPOUT}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

Embedding size: 256
Hidden size   : 512
Num layers    : 1
Dropout       : 0.3
Total parameters: 15,509,632
RecurrentNMT(
  (encoder): BiLSTMEncoder(
    (embedding): ManualEmbedding()
    (dropout): Dropout(p=0.3, inplace=False)
    (forward_layers): ModuleList(
      (0): ManualLSTMLayer(
        (cell): ManualLSTMCell()
      )
    )
    (backward_layers): ModuleList(
      (0): ManualLSTMLayer(
        (cell): ManualLSTMCell()
      )
    )
    (init_hidden): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): AttentiveLSTMDecoder(
    (embedding): ManualEmbedding()
    (dropout): Dropout(p=0.3, inplace=False)
    (attention): AdditiveAttention(
      (decoder_proj): Linear(in_features=512, out_features=512, bias=False)
      (encoder_proj): Linear(in_features=1024, out_features=512, bias=False)
      (energy): Linear(in_features=512, out_features=1, bias=False)
    )
    (cells): ModuleList(
      (0): ManualLSTMCell()
    )
    (output): Linear(in_features=1792,

In [4]:
# Forward pass sanity check
src_ids, dec_input, targets = next(iter(train_loader))
src_ids, dec_input = src_ids.to(DEVICE), dec_input.to(DEVICE)
with torch.no_grad():
    logits, _, _, lstm_attn = model(src_ids, dec_input)

print('src_ids:', src_ids.shape)
print('dec_input:', dec_input.shape)
print('logits:', logits.shape)       # (batch, tgt_len, target_vocab)
print('attention:', lstm_attn.shape) # (batch, 1, tgt_len, src_len)
print('targets:', targets.shape)

src_ids: torch.Size([32, 16])
dec_input: torch.Size([32, 11])
logits: torch.Size([32, 11, 3200])
attention: torch.Size([32, 1, 11, 16])
targets: torch.Size([32, 11])


## 3. Train

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_LEARNING_RATE)

train_losses, val_losses = train(
    model, train_loader, val_loader, optimizer,
    pad_id=PAD_ID, max_epochs=LSTM_MAX_EPOCHS,
    device=DEVICE, checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_name='best_lstm.pt',
)

Epoch  1/10  train_loss=3.1817  val_loss=2.3576
  Checkpoint saved -> d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\best_lstm.pt
Epoch  2/10  train_loss=2.0704  val_loss=1.8813
  Checkpoint saved -> d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\best_lstm.pt
Epoch  3/10  train_loss=1.5591  val_loss=1.6201
  Checkpoint saved -> d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\best_lstm.pt
Epoch  4/10  train_loss=1.1361  val_loss=1.4410
  Checkpoint saved -> d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\best_lstm.pt
Epoch  5/10  train_loss=0.7809  val_loss=1.3448
  Checkpoint saved -> d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\best_lstm.pt
Epoch  6/10  train_loss=0.5095  val_loss=1.3027
  Checkpoint saved -> d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\best_lstm.pt
Epoch  7/10  train_loss=0.3296  val_loss=1.3202
Epoch  8/10  train_loss=0.2221  va

In [6]:
fig = plot_loss_curves(train_losses, val_losses)
plt.savefig('lstm_loss_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('Loss curve saved to lstm_loss_curves.png')

Loss curve saved to lstm_loss_curves.png


C:\Users\Wind\AppData\Local\Temp\ipykernel_15352\125155201.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Save and Load Checkpoint

In [7]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
save_checkpoint(
    model, optimizer, LSTM_MAX_EPOCHS, val_losses[-1],
    os.path.join(CHECKPOINT_DIR, 'final_lstm.pt'),
)

epoch, val_loss = load_checkpoint(
    model, optimizer,
    os.path.join(CHECKPOINT_DIR, 'best_lstm.pt'),
    DEVICE,
)

  Checkpoint saved -> d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\final_lstm.pt
Loaded checkpoint from d:\Term 8\NLP\NLP-Projects\assigment_3\nmt_transformer\../checkpoints\best_lstm.pt  (epoch 6, val_loss 1.3027)


## 5. Greedy vs. Beam Search

In [8]:
test_sentences = [
    'je suis dure .',
    'il est tres intelligent .',
    'bonjour , comment ca va ?',
    'elle aime lire des livres .',
    "nous allons a l ' ecole demain .",
]

print(f"{'French':<40} {'Greedy':<30} Beam (k=4)")
print('-' * 100)
for fr in test_sentences:
    greedy_out, _ = lstm_greedy_decode(
        model, fr, src_tokenizer, tgt_tokenizer,
        MAX_SEQ_LEN, BOS_ID, EOS_ID, PAD_ID, DEVICE,
    )
    beam_out, _, _ = lstm_beam_search(
        model, fr, src_tokenizer, tgt_tokenizer,
        BEAM_SIZE, MAX_SEQ_LEN, BOS_ID, EOS_ID, PAD_ID, DEVICE,
    )
    print(f'{fr:<40} {clean(greedy_out):<30} {clean(beam_out)}')

French                                   Greedy                         Beam (k=4)
----------------------------------------------------------------------------------------------------
je suis dure .                           i m tough .                    i m tough .
il est tres intelligent .                he is very intelligent .       he is very intelligent .
bonjour , comment ca va ?                i m fine aren t you ?          i m fine aren t you ?
elle aime lire des livres .              she is fond of taking a wall . she is fond of c ake .
nous allons a l ' ecole demain .         we re going to leave tomorrow . we re going to leave tomorrow .


## 6. BLEU Score

In [9]:
print('Computing BLEU on test set ...')
bleu_result, hypotheses, references = compute_bleu_dataset(
    model, ds['test'], src_tokenizer, tgt_tokenizer,
    beam_size=BEAM_SIZE, max_len=MAX_SEQ_LEN,
    bos_id=BOS_ID, eos_id=EOS_ID, pad_id=PAD_ID, device=DEVICE,
    decode_fn=lstm_beam_search,
)
print(f'Test BLEU = {bleu_result.score:.2f}')
print(bleu_result)

print('Sample predictions:')
for i in range(5):
    print(f'REF: {references[i]}')
    print(f'HYP: {clean(hypotheses[i])}')
    print()

Computing BLEU on test set ...
Test BLEU = 0.06
BLEU = 0.06 14.3/0.0/0.0/0.0 (BP = 1.000 ratio = 1.162 hyp_len = 3367 ref_len = 2898)
Sample predictions:
REF: i m trustworthy .
HYP: i m being blackmailed .

REF: i m not miserable .
HYP: i m not available .

REF: i m going to take my car .
HYP: i m going to drive my car .

REF: he s a cat lover .
HYP: he is able to cats .

REF: i m happy with that .
HYP: i m happy with it .



## 7. Attention Visualization

In [14]:
src_text = 'je suis dure .'
translation, pred_tokens, lstm_attn = lstm_beam_search(
    model, src_text, src_tokenizer, tgt_tokenizer,
    BEAM_SIZE, MAX_SEQ_LEN, BOS_ID, EOS_ID, PAD_ID, DEVICE,
)
print(f'FR: {src_text}')
print(f'EN: {clean(translation)}')

src_ids_list = src_tokenizer.encode(src_text, add_special_tokens=False)
src_ids_list = src_ids_list[: MAX_SEQ_LEN - 1] + [EOS_ID]
src_labels = ids_to_tokens(src_ids_list, src_tokenizer)
tgt_labels = ids_to_tokens(pred_tokens, tgt_tokenizer)

if lstm_attn is not None:
    attn_slice = lstm_attn[:, :, :len(tgt_labels), :len(src_labels)]
    fig = plot_attention(attn_slice, src_labels, tgt_labels,
                         title='LSTM Additive Attention')
    plt.savefig('lstm_attention.png', dpi=100, bbox_inches='tight')
    plt.show()

print('Attention plot saved to lstm_attention.png')

FR: je suis dure .
EN: i m tough .
Attention plot saved to lstm_attention.png


C:\Users\Wind\AppData\Local\Temp\ipykernel_15352\2209812785.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Live Translation

Use the dedicated script in the terminal:

```bash
python live_lstm_test.py
python live_lstm_test.py --sentence "je suis dure ."
python live_lstm_test.py --show_attn
```